The following additional libraries are needed to run this
notebook. Note that running on Colab is experimental, please report a Github
issue if you have any problem.

# 循环神经网络的简洁实现
:label:`sec_rnn-concise`

虽然 :numref:`sec_rnn_scratch`
对了解循环神经网络的实现方式具有指导意义，但并不方便。
本节将展示如何使用深度学习框架的高级API提供的函数更有效地实现相同的语言模型。
我们仍然从读取时光机器数据集开始。


In [36]:
import re
import requests
import torch
from torch.utils.data import DataLoader, TensorDataset

def load_data_time_machine(batch_size, num_steps, max_tokens=20000):
    url = "https://d2l-data.s3-accelerate.amazonaws.com/timemachine.txt"
    text = requests.get(url).text

    # 清洗文本：只保留字母，其他变空格，小写
    text = re.sub("[^A-Za-z]+", " ", text).lower()
    text = text[:max_tokens]

    # 字符级词表
    chars = sorted(list(set(text)))
    idx_to_token = ["<unk>"] + chars
    token_to_idx = {ch: i for i, ch in enumerate(idx_to_token)}

    class Vocab:
        def __len__(self):
            return len(idx_to_token)

        def __getitem__(self, tokens):
            if isinstance(tokens, (list, tuple)):
                return [self.__getitem__(token) for token in tokens]
            return token_to_idx.get(tokens, 0)

        def to_tokens(self, indices):
            if isinstance(indices, (list, tuple)):
                return [idx_to_token[int(i)] for i in indices]
            return idx_to_token[int(indices)]

    vocab = Vocab()

    corpus = torch.tensor([vocab[ch] for ch in text], dtype=torch.long)

    # 构造 X 和 Y，Y 是 X 向后移动一位
    num_subseqs = (len(corpus) - 1) // num_steps

    Xs = []
    Ys = []

    for i in range(num_subseqs):
        start = i * num_steps
        Xs.append(corpus[start:start + num_steps])
        Ys.append(corpus[start + 1:start + num_steps + 1])

    Xs = torch.stack(Xs)
    Ys = torch.stack(Ys)

    dataset = TensorDataset(Xs, Ys)

    data_iter = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False
    )

    return data_iter, vocab

In [50]:
import torch
from torch import nn
from torch.nn import functional as F

batch_size, num_steps = 32, 35
train_iter, vocab = load_data_time_machine(batch_size, num_steps)

## [**定义模型**]

高级API提供了循环神经网络的实现。
我们构造一个具有256个隐藏单元的单隐藏层的循环神经网络层`rnn_layer`。
事实上，我们还没有讨论多层循环神经网络的意义（这将在 :numref:`sec_deep_rnn`中介绍）。
现在仅需要将多层理解为一层循环神经网络的输出被用作下一层循环神经网络的输入就足够了。


In [51]:
num_hiddens = 128
# rnn_layer = nn.RNN(len(vocab), num_hiddens)
rnn_layer = nn.RNN(
    input_size=len(vocab),
    hidden_size=num_hiddens,
    num_layers=3,
    dropout=0.2
)

我们(**使用张量来初始化隐状态**)，它的形状是（隐藏层数，批量大小，隐藏单元数）。


In [52]:
state = torch.zeros((3, batch_size, num_hiddens))
state.shape

torch.Size([3, 32, 128])

[**通过一个隐状态和一个输入，我们就可以用更新后的隐状态计算输出。**]
需要强调的是，`rnn_layer`的“输出”（`Y`）不涉及输出层的计算：
它是指每个时间步的隐状态，这些隐状态可以用作后续输出层的输入。


In [53]:
X = torch.rand(size=(num_steps, batch_size, len(vocab)))
Y, state_new = rnn_layer(X, state)
Y.shape, state_new.shape

(torch.Size([35, 32, 128]), torch.Size([3, 32, 128]))

与 :numref:`sec_rnn_scratch`类似，
[**我们为一个完整的循环神经网络模型定义了一个`RNNModel`类**]。
注意，`rnn_layer`只包含隐藏的循环层，我们还需要创建一个单独的输出层。


In [54]:
class RNNModel(nn.Module):
    """简单循环神经网络语言模型，只考虑普通 RNN"""
    def __init__(self, rnn_layer, vocab_size):
        super().__init__()

        self.rnn = rnn_layer
        self.vocab_size = vocab_size
        self.num_hiddens = self.rnn.hidden_size

        # 把 RNN 的隐藏状态映射到词表大小
        self.linear = nn.Linear(self.num_hiddens, self.vocab_size)

    def forward(self, inputs, state):
        # inputs shape: (batch_size, num_steps)

        # 转置后 shape: (num_steps, batch_size)
        # one-hot 后 shape: (num_steps, batch_size, vocab_size)
        X = F.one_hot(inputs.T.long(), self.vocab_size).float()

        # Y shape: (num_steps, batch_size, num_hiddens)
        # state shape: (num_layers, batch_size, num_hiddens)
        Y, state = self.rnn(X, state)

        # reshape 后 shape: (num_steps * batch_size, num_hiddens)
        # output shape: (num_steps * batch_size, vocab_size)
        output = self.linear(Y.reshape(-1, self.num_hiddens))

        return output, state

    def begin_state(self, device, batch_size=1):
        # 普通 RNN 的初始 hidden state
        return torch.zeros(
            self.rnn.num_layers,
            batch_size,
            self.num_hiddens,
            device=device
        )

## 训练与预测

在训练模型之前，让我们[**基于一个具有随机权重的模型进行预测**]。


In [55]:
import torch
from torch.nn import functional as F


def try_gpu():
    """如果有 GPU，则使用 GPU，否则使用 CPU"""
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def predict_ch8(prefix, num_preds, net, vocab, device):
    """在 prefix 后面生成新字符，不依赖 d2l"""
    net.eval()

    state = net.begin_state(batch_size=1, device=device)
    outputs = [vocab[prefix[0]]]

    def get_input():
        return torch.tensor([[outputs[-1]]], device=device)

    # 预热：用 prefix 中已有字符更新 RNN 状态
    for y in prefix[1:]:
        _, state = net(get_input(), state)
        outputs.append(vocab[y])

    # 逐字符预测
    for _ in range(num_preds):
        y_hat, state = net(get_input(), state)
        next_token = int(y_hat.argmax(dim=1).reshape(1))
        outputs.append(next_token)

    return ''.join(vocab.to_tokens(outputs))

In [59]:
device = try_gpu()
net = RNNModel(rnn_layer, vocab_size=len(vocab))
net = net.to(device)
predict_ch8('time traveller', 10, net, vocab, device)

'time travellerttygaytlla'

很明显，这种模型根本不能输出好的结果。
接下来，我们使用 :numref:`sec_rnn_scratch`中
定义的超参数调用`train_ch8`，并且[**使用高级API训练模型**]。


In [60]:
import math
import torch
from torch import nn

num_epochs = 1000

loss = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(net.parameters(), lr=1)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=100,
    gamma=0.6
)

for epoch in range(num_epochs):
    state = None
    total_loss = 0.0
    total_tokens = 0

    net.train()

    for X, Y in train_iter:
        X, Y = X.to(device), Y.to(device)

        if state is None or state.shape[1] != X.shape[0]:
            state = net.begin_state(
                device=device,
                batch_size=X.shape[0]
            )
        else:
            # 截断梯度，但保留 hidden state 的数值
            state = state.detach()

        y = Y.T.reshape(-1)

        y_hat, state = net(X, state)

        l = loss(y_hat, y.long())

        optimizer.zero_grad()
        l.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        optimizer.step()

        total_loss += l.item() * y.numel()
        total_tokens += y.numel()

    scheduler.step()

    ppl = math.exp(total_loss / total_tokens)

    if (epoch + 1) % 10 == 0:
        print(
            f"epoch {epoch + 1}, "
            f"perplexity {ppl:.1f}, "
            f"{predict_ch8('time traveller', 50, net, vocab, device)}"
        )

epoch 10, perplexity 3.2, time traveller s i at the time traveller s i at the time travell
epoch 20, perplexity 3.0, time traveller s a man who saw and the time traveller s a man wh
epoch 30, perplexity 3.0, time traveller s a minute s passed the time traveller s a minute
epoch 40, perplexity 3.0, time traveller he s of the time traveller he s of the time trave
epoch 50, perplexity 3.0, time traveller have nor of a well he surt the time traveller hav
epoch 60, perplexity 3.0, time traveller s a some said the medical man was some speet the 
epoch 70, perplexity 3.0, time traveller s a simill the time traveller s and with a see sh
epoch 80, perplexity 3.0, time traveller s a mathematical man and the time traveller s a m
epoch 90, perplexity 3.0, time traveller s i said the medical man a cube then the time tra
epoch 100, perplexity 2.9, time traveller s abself and the time traveller s abself and the 
epoch 110, perplexity 2.5, time traveller smiled in the time traveller smiled in the time

In [61]:
print(predict_ch8('time traveller', 50, net, vocab, device))
print(predict_ch8('traveller', 50, net, vocab, device))
print(predict_ch8('love you', 50, net, vocab, device))

time traveller smiled are you sure we can sound been you see he 
traveller smiled are you sure we can sound been you see he 
love you cannot move about in all directions that s a simp


与上一节相比，由于深度学习框架的高级API对代码进行了更多的优化，
该模型在较短的时间内达到了较低的困惑度。

## 小结

* 深度学习框架的高级API提供了循环神经网络层的实现。
* 高级API的循环神经网络层返回一个输出和一个更新后的隐状态，我们还需要计算整个模型的输出层。
* 相比从零开始实现的循环神经网络，使用高级API实现可以加速训练。

## 练习

1. 尝试使用高级API，能使循环神经网络模型过拟合吗？
1. 如果在循环神经网络模型中增加隐藏层的数量会发生什么？能使模型正常工作吗？
1. 尝试使用循环神经网络实现 :numref:`sec_sequence`的自回归模型。


[Discussions](https://discuss.d2l.ai/t/2106)
